# COPD Automated Clinical Data Pipeline

## Overview
This pipeline automatically processes raw COPD patient data to support 
clinical decision-making. It is designed to run on any new dataset and 
will automatically:

- Detect and flag data quality issues (duplicates, missing values, 
  data entry errors, misaligned rows)
- Fix issues where safe to do so automatically
- Run anomaly detection to identify patients with unusual clinical readings
- Generate a professional HTML report for the clinical team

This pipeline is built with clinical safety in mind — patient data is 
never silently removed. All issues are flagged transparently for 
clinical review.

**Dataset:** COPD patient monitoring data
**Output:** HTML clinical report with data quality alerts and priority patients
**Author:** Jerin
**Date:** May 2026

## Step 1: Setup and Imports
Import all required libraries and configure API access.

In [1]:
# Import libraries
import pandas as pd
import sqlite3
from datetime import datetime
import os
from dotenv import load_dotenv
from google import genai

In [2]:
# API Key setup
# store your API key in a .env file and load it using dotenv
# never hardcode your actual API key here before pushing to GitHub

os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
api_key = os.environ["GEMINI_API_KEY"]

# Connect to Gemini
client = genai.Client(api_key=api_key)

print("Setup complete!")

Setup complete!


## Step 2: Load Raw Data
Load the raw COPD dataset into a SQLite database for querying.
The raw data is intentionally kept unmodified at this stage —
all issues will be detected and flagged in Step 3.

In [3]:
# Load raw data
df = pd.read_csv("COPD.csv")
print(f"Loaded {len(df)} rows and {len(df.columns)} columns")

# Create database connection
conn = sqlite3.connect('copd_raw.db')

# Save to database
df.to_sql('patients_raw', conn, if_exists='replace', index=False)
print("Raw data saved to database successfully")

Loaded 101 rows and 24 columns
Raw data saved to database successfully


## Step 3: Data Quality Checks
Automatically detect data quality issues in the raw dataset.
All issues are flagged for clinical review — no data is silently removed.

Checks performed:
- Duplicate patient IDs
- Missing critical clinical values
- Data entry errors (values outside valid clinical ranges)
- Misaligned rows (values shifted across columns)

In [4]:
# STEP 3: DATA QUALITY CHECKS

# Check 1: Checking for duplicates using ID
duplicates = pd.read_sql("""
    SELECT ID, COUNT(*) as id_count 
    FROM patients_raw 
    GROUP BY ID 
    HAVING id_count > 1
""", conn)

# Flagging nulls in all columns that directly affect analysis
critical_missing = pd.read_sql("""
    SELECT ID, MWT1, MWT2, MWT1Best, FEV1, FVC, CAT
    FROM patients_raw
    WHERE MWT1 IS NULL OR MWT2 IS NULL OR MWT1Best IS NULL
    OR FEV1 IS NULL OR FVC IS NULL OR CAT IS NULL
    OR AGE IS NULL OR gender IS NULL
""", conn)

# Check 3: Data entry errors of columns that can affect analysis
data_entry_errors = pd.read_sql("""
    SELECT ID, CAT 
    FROM patients_raw 
    WHERE CAT > 40 OR CAT < 0
""", conn)

# Check 4: Checking multiple columns for misallignment or invalid values
misaligned_rows = pd.read_sql("""
    SELECT ID, HAD, SGRQ, gender, copd, Diabetes, smoking
    FROM patients_raw 
    WHERE HAD > 42 OR HAD < 0
    OR gender NOT IN (0, 1)
    OR copd NOT IN (1, 2, 3, 4)
    OR Diabetes NOT IN (0, 1)
    OR smoking NOT IN (1, 2)
""", conn)

# Summary
print("Data Quality Summary:")
print(f"{'='*40}")
print(f"Duplicates found:        {len(duplicates)}")
print(f"Missing MWT values:      {len(critical_missing)}")
print(f"Data entry errors:       {len(data_entry_errors)}")
print(f"Misaligned rows:         {len(misaligned_rows)}")
print(f"{'='*40}")
total_issues = len(duplicates) + len(critical_missing) + len(data_entry_errors) + len(misaligned_rows)
print(f"Total issues found:      {total_issues}")

Data Quality Summary:
Duplicates found:        4
Missing MWT values:      2
Data entry errors:       1
Misaligned rows:         1
Total issues found:      8


## Step 4: Auto-Fix Where Possible
Automatically fix data issues where it is safe to do so.

Fix applied:
- MWT1Best: If missing but MWT1 or MWT2 available, use the available value
- All other issues are flagged only — never silently changed

Note: Unnecessary columns (unnamed index, COPDSEVERITY) are dropped 
at this stage as they add no analytical value.

In [5]:
# STEP 4: AUTO-FIX WHERE POSSIBLE

# Load data into pandas for fixing
df = pd.read_sql("SELECT * FROM patients_raw", conn)

# Drop unnecessary columns if they exist
cols_to_drop = ['Unnamed: 0', 'COPDSEVERITY']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"Dropped column: {col}")

# Fix MWT1Best where missing but MWT1 or MWT2 available
df.loc[df['MWT1Best'].isna() & df['MWT1'].notna(), 'MWT1Best'] = \
    df.loc[df['MWT1Best'].isna() & df['MWT1'].notna(), 'MWT1']

df.loc[df['MWT1Best'].isna() & df['MWT2'].notna(), 'MWT1Best'] = \
    df.loc[df['MWT1Best'].isna() & df['MWT2'].notna(), 'MWT2']

# Update database with fixed values
df.to_sql('patients_raw', conn, if_exists='replace', index=False)

print("\nAuto-Fix Summary:")
print(f"{'='*40}")
print(f"MWT1Best auto-fix complete")
print(f"Remaining missing MWT1Best: {df['MWT1Best'].isna().sum()}")

Dropped column: Unnamed: 0
Dropped column: COPDSEVERITY

Auto-Fix Summary:
MWT1Best auto-fix complete
Remaining missing MWT1Best: 1


## Step 5: Anomaly Detection
Detect patients with unusual clinical readings using group-based 
anomaly detection.

Method: Values outside 2 standard deviations of their severity 
group mean are flagged as anomalies.

Columns checked:
- FEV1 (lung function)
- FVC (lung capacity)
- MWT1Best (physical performance)
- CAT (symptom burden)

In [6]:
# STEP 5: ANOMALY DETECTION

columns_to_check = ['FEV1', 'FVC', 'MWT1Best', 'CAT']

for column in columns_to_check:
    # Calculate group mean and std
    df[f'{column}_group_mean'] = df.groupby('copd')[column].transform('mean')
    df[f'{column}_group_std'] = df.groupby('copd')[column].transform('std')
    
    # Flag anomalies outside 2 standard deviations
    df[f'{column}_anomaly'] = (
        (df[column] < df[f'{column}_group_mean'] - 2 * df[f'{column}_group_std']) |
        (df[column] > df[f'{column}_group_mean'] + 2 * df[f'{column}_group_std'])
    )

# Combined anomaly flag
df['any_anomaly'] = (
    df['FEV1_anomaly'] | 
    df['FVC_anomaly'] | 
    df['MWT1Best_anomaly'] | 
    df['CAT_anomaly']
)

# Anomaly count
df['anomaly_count'] = (
    df['FEV1_anomaly'].astype(int) + 
    df['FVC_anomaly'].astype(int) + 
    df['MWT1Best_anomaly'].astype(int) + 
    df['CAT_anomaly'].astype(int)
)

# Priority level
def priority(count):
    if count == 0:
        return 'NORMAL'
    elif count >= 3:
        return 'CRITICAL'
    elif count == 2:
        return 'WARNING'
    else:
        return 'MONITOR'

df['priority_level'] = df['anomaly_count'].apply(priority)

# Save results to database
df.to_sql('patients_raw', conn, if_exists='replace', index=False)

print("\nAnomaly Detection Summary:")
print(f"{'='*40}")
print(f"Total patients:          {len(df)}")
print(f"Normal patients:         {(df['priority_level'] == 'NORMAL').sum()}")
print(f"Monitor patients:        {(df['priority_level'] == 'MONITOR').sum()}")
print(f"Warning patients:        {(df['priority_level'] == 'WARNING').sum()}")
print(f"Critical patients:       {(df['priority_level'] == 'CRITICAL').sum()}")
print(f"{'='*40}")
print(f"Total flagged:           {df['any_anomaly'].sum()}")


Anomaly Detection Summary:
Total patients:          101
Normal patients:         88
Monitor patients:        12
Warning patients:        0
Critical patients:       1
Total flagged:           13


## Step 6: Generate Clinical Report
Automatically generate a professional HTML clinical report containing:
- Data quality alerts
- KPI summary
- Anomaly breakdown by measure
- Flagged patients by severity group
- Priority patients table
- AI generated narrative summary

In [7]:
# STEP 6: GENERATE CLINICAL REPORT

def generate_report():
    today = datetime.now().strftime('%d %B %Y')
    
    # KEY STATS
    total_patients = pd.read_sql("SELECT COUNT(*) as count FROM patients_raw", conn).iloc[0]['count']
    flagged_patients = pd.read_sql("SELECT COUNT(*) as count FROM patients_raw WHERE any_anomaly = 1", conn).iloc[0]['count']
    critical_patients = pd.read_sql("SELECT COUNT(*) as count FROM patients_raw WHERE priority_level = 'CRITICAL'", conn).iloc[0]['count']
    
    # HTML HEADER
    html = f"""
    <html>
    <head>
        <title>COPD Patient Monitoring Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; background-color: #f9f9f9; }}
            h1 {{ color: #2c3e50; }}
            h2 {{ color: #2980b9; border-bottom: 2px solid #2980b9; padding-bottom: 5px; }}
            table {{ border-collapse: collapse; margin-bottom: 20px; }}
            th, td {{ padding: 10px; border: 1px solid #ddd; }}
            .alert {{ background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px; margin: 10px 0; border-radius: 5px; }}
            .critical-alert {{ background-color: #ffcccc; border-left: 4px solid #e74c3c; padding: 10px; margin: 10px 0; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>COPD Patient Monitoring Report</h1>
        <p>Generated: {today}</p>
    """

    # DATA QUALITY ALERTS
    html += """<h2>⚠️ Data Quality Alerts</h2>"""
    
    if len(duplicates) > 0:
        dup_ids = ', '.join(map(str, duplicates['ID'].tolist()))
        html += f"""
        <div class="critical-alert">
            <b>🔴 Duplicate Patient IDs Found: {len(duplicates)}</b><br>
            Affected IDs: {dup_ids}<br>
            Action Required: Clinical team must verify these are not different patients sharing the same ID.
        </div>
        """
    
    if len(critical_missing) > 0:
        missing_ids = ', '.join(map(str, critical_missing['ID'].tolist()))
        html += f"""
        <div class="alert">
            <b>🟡 Missing Critical Values: {len(critical_missing)} patients</b><br>
            Affected IDs: {missing_ids}<br>
            Action Required: Walk test data missing — anomaly detection for MWT1Best may be incomplete.
        </div>
        """
    
    if len(data_entry_errors) > 0:
        error_ids = ', '.join(map(str, data_entry_errors['ID'].tolist()))
        html += f"""
        <div class="critical-alert">
            <b>🔴 Data Entry Errors: {len(data_entry_errors)} patients</b><br>
            Affected IDs: {error_ids}<br>
            Action Required: CAT score outside valid range (0-40) — verify and correct before clinical decisions.
        </div>
        """
    
    if len(misaligned_rows) > 0:
        misaligned_ids = ', '.join(map(str, misaligned_rows['ID'].tolist()))
        html += f"""
        <div class="critical-alert">
            <b>🔴 Misaligned Data Detected: {len(misaligned_rows)} patients</b><br>
            Affected IDs: {misaligned_ids}<br>
            Action Required: Column values appear shifted — manual review required.
        </div>
        """

    # KPI SUMMARY
    html += f"""
        <h2>Summary</h2>
        <table style="width:50%;">
            <tr style="background-color:#2980b9; color:white;">
                <th>Metric</th>
                <th>Value</th>
            </tr>
            <tr>
                <td>Total Patients Monitored</td>
                <td><b>{total_patients}</b></td>
            </tr>
            <tr style="background-color:#fff3cd;">
                <td>Total Flagged Patients</td>
                <td><b>{flagged_patients}</b></td>
            </tr>
            <tr style="background-color:#ffcccc;">
                <td>Critical Patients</td>
                <td><b>{critical_patients}</b></td>
            </tr>
        </table>
    """

    # ANOMALY BY MEASURE
    measures = {
        'FEV1': int(df['FEV1_anomaly'].sum()),
        'FVC': int(df['FVC_anomaly'].sum()),
        'MWT1Best': int(df['MWT1Best_anomaly'].sum()),
        'CAT': int(df['CAT_anomaly'].sum())
    }
    
    html += """
        <h2>Anomaly Breakdown by Measure</h2>
        <table style="width:50%;">
            <tr style="background-color:#2980b9; color:white;">
                <th>Measure</th>
                <th>Anomalies Found</th>
            </tr>
    """
    
    for measure, count in measures.items():
        colour = '#fff3cd' if count == max(measures.values()) else 'white'
        html += f"""
            <tr style="background-color:{colour};">
                <td>{measure}</td>
                <td>{count}</td>
            </tr>
        """
    
    html += "</table>"

    # ANOMALY BY SEVERITY GROUP
    severity_breakdown = pd.read_sql("""
        SELECT copd, COUNT(*) as flagged_patients
        FROM patients_raw 
        WHERE any_anomaly = 1 
        GROUP BY copd
        ORDER BY copd
    """, conn)
    
    severity_labels = {1: 'MILD', 2: 'MODERATE', 3: 'SEVERE', 4: 'VERY SEVERE'}
    
    html += """
        <h2>Flagged Patients by Severity Group</h2>
        <table style="width:50%;">
            <tr style="background-color:#2980b9; color:white;">
                <th>Severity Group</th>
                <th>Flagged Patients</th>
            </tr>
    """
    
    for _, row in severity_breakdown.iterrows():
        html += f"""
            <tr>
                <td>{severity_labels[row['copd']]}</td>
                <td>{int(row['flagged_patients'])}</td>
            </tr>
        """
    
    html += "</table>"

    # PRIORITY PATIENTS TABLE
    priority_patients = pd.read_sql("""
        SELECT ID, copd, priority_level, anomaly_count, 
               FEV1, MWT1Best, CAT
        FROM patients_raw 
        WHERE any_anomaly = 1 
        ORDER BY anomaly_count DESC
    """, conn)
    
    html += """
        <h2>Priority Patients - Flagged for Clinical Review</h2>
        <table style="width:80%;">
            <tr style="background-color:#2980b9; color:white;">
                <th>ID</th>
                <th>Severity</th>
                <th>Priority</th>
                <th>Anomaly Count</th>
                <th>FEV1</th>
                <th>MWT1Best</th>
                <th>CAT</th>
            </tr>
    """
    
    for _, row in priority_patients.iterrows():
        if row['priority_level'] == 'CRITICAL':
            colour = '#ffcccc'
        elif row['priority_level'] == 'WARNING':
            colour = '#fff3cd'
        else:
            colour = 'white'
        
        html += f"""
            <tr style="background-color:{colour};">
                <td>{int(row['ID'])}</td>
                <td>{severity_labels[int(row['copd'])]}</td>
                <td><b>{row['priority_level']}</b></td>
                <td>{int(row['anomaly_count'])}</td>
                <td>{row['FEV1']}</td>
                <td>{row['MWT1Best']}</td>
                <td>{int(row['CAT'])}</td>
            </tr>
        """
    
    html += "</table>"

    # AI NARRATIVE SUMMARY 
    
    # get the highest priority patient dynamically
    top_patient = priority_patients.iloc[0]
    
    summary_prompt = f"""
    You are a clinical data analyst reporting to a medical team.
    Write a brief 3-4 sentence plain English summary of these COPD patient monitoring results:

    - Total patients monitored: {total_patients}
    - Total flagged patients: {flagged_patients}
    - Critical patients: {critical_patients}
    - Data quality issues: {len(duplicates)} duplicate IDs, {len(critical_missing)} missing values, 
      {len(data_entry_errors)} data entry errors, {len(misaligned_rows)} misaligned rows
    - Measure with most anomalies: MWT1Best
    - Severity group with most anomalies: MODERATE
    - Highest priority patient: ID {int(top_patient['ID'])} ({int(top_patient['anomaly_count'])} anomalies)

    Keep it professional, concise and suitable for a clinical team.
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=summary_prompt
    )
    
    html += f"""
        <h2>AI Generated Summary</h2>
        <div style="background-color:#eaf4fb; padding:15px; border-left:4px solid #2980b9; border-radius:5px;">
            {response.text}
        </div>
    </body>
    </html>"""
    
    return html

### Run and Save Report
Execute the pipeline and save the output as a professional HTML report.

In [8]:
# Run and Save Report
report = generate_report()

with open('copd_pipeline_report.html', 'w', encoding='utf-8') as f:
    f.write(report)

print("Report saved successfully!")

Report saved successfully!
